# Variable/Constants & Imports

In [59]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm, uniform, chi2
from collections.abc import Callable
import plotly.graph_objects as go
import inspect

n_bins = 300
N = 31851
TAU_MU = 2.1969811e-6
exp_args = {"a": 1, "A": 0, "tau": TAU_MU}

# Functions

In [60]:

def exp( x , N ,a, A, tau):
    return a*N*expon.cdf(x , A , tau) 

def sturges( N: int):
    return int(np.round( 1 + 3.322 * np.log10(N)))

def exp_unif(x, a, b, tau, A):
    return a * N * (expon.cdf(x, A, tau) + b * N * uniform.cdf(x, 0, 8e-6))


def exp_fit(cdf, a, A, tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, A, tau )
    n.fixed["A"] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def exp_unif_fit(cdf, a, b, tau, A, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, b, tau, A )
    # n.fixed['N'] = True
    n.fixed['A'] = True
    # n.limits['b'] = (0, 1)
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n


#! Funzione per generalizzare la funzione usando la lunghezza del dataset
def function_generator_with_variable_N(func: Callable, N: int):
    sig = inspect.signature(func)
    
    if "x" not in sig.parameters or "N" not in sig.parameters:
        raise SyntaxError("function defined wrong: it needs an x as first parameter and N as other parameter")
    def wrapper(x, *args, **kwargs):
        return func(x, N, *args, **kwargs)

    wrapper.__signature__ = inspect.Signature(
        [inspect.Parameter('x', inspect.Parameter.POSITIONAL_OR_KEYWORD)] +
        list(inspect.signature(func).parameters.values())[2:]
    )

    return wrapper

def dataset_analysis(dataset: np.ndarray , creator: Callable, bins: int , args: dict) -> Minuit:
    model_function = function_generator_with_variable_N( creator , len(dataset))
    count, edges = np.histogram( dataset , bins=bins) 
    cost = ExtendedBinnedNLL(count, edges, model_function)

    minuit_element =  Minuit(cost, *args.values())
    
    if "A" in args.keys():
        minuit_element.fixed["A"] = True

    return minuit_element

def end(m:Minuit) -> None:
    m.migrad()
    m.hesse()
    display(m)

# Dataset

In [61]:
data_0 = np.genfromtxt("Data/timestamp/21_01_2024_17_42.csv", delimiter=',')
data_1 = np.genfromtxt("Data/timestamp/23_01_2026_17_31.csv", delimiter=',')
data_2 = np.genfromtxt("Data/timestamp/29_01_2026_17_20.csv", delimiter=',')
data_3 = np.genfromtxt("Data/timestamp/02_02_2026_17_14.csv", delimiter=',')
data_4 = np.genfromtxt("Data/timestamp/05_02_2026_18_17.csv", delimiter=',')
data = np.concatenate((data_0, data_1, data_2, data_3, data_4 ))


# Analysis

## Hists

In [62]:

# data_2 = data_2[(data_2 > 0.6e-6) & (data_2 < 1.8e-6)]
count, edges = np.histogram(data, bins=n_bins , density=False)
count_1, edges_1 = np.histogram(data_1, bins=n_bins , density=False)
count_2, edges_2 = np.histogram(data_2, bins=n_bins , density=False)
count_3, edges_3 = np.histogram(data_3, bins=n_bins , density=False)
count_4, edges_4 = np.histogram(data_4, bins=n_bins , density=False)
print (len(data), len(data_1))
fig = go.Figure()

fig.add_trace(go.Bar(x=edges[:-1],  y=count   / len(data),   name='total',        width=np.diff(edges)))
fig.add_trace(go.Bar(x=edges_1[:-1], y=count_1 / len(data_1), name='23/01/2026',   width=np.diff(edges_1)))
fig.add_trace(go.Bar(x=edges_2[:-1], y=count_2 / len(data_2), name='29/01/2026',   width=np.diff(edges_2)))
fig.add_trace(go.Bar(x=edges_3[:-1], y=count_3 / len(data_3), name='02/02/2026',   width=np.diff(edges_3)))
fig.add_trace(go.Bar(x=edges_4[:-1], y=count_4 / len(data_4), name='05/02/2026',   width=np.diff(edges_4)))
fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

31851 9400


In [63]:
k = dataset_analysis( data_1 , exp , 69 , exp_args)
end(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 216.6 (χ²/ndof = 3.2)      │              Nfcn = 38               │
│ EDM = 1.03e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.027   │   0.011   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.230e-6  │ 0.029e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │           a           A         tau │
├─────┼─────────────────────────────────────┤
│   a │    0.000114           0 37.2538e-12 │
│   A │           0           0           0 │
│ tau │ 37.2538e-12           0    8.32e-16 │
└─────┴─────────────────────────────────────┘

## Full interpolation 

In [64]:
n_bins = 53
exp_args = {"a": 1, "A": 0, "tau": TAU_MU}

data_refact = data[(data > 1e-7)&(data < 7.1e-6)]## forse si può anche non tagliare dal basso


n = dataset_analysis( data_refact , exp , bins = n_bins , args=exp_args)

end(n)


p_value = 1 - chi2.cdf(n.fval, df=n.ndof)
print(p_value)



┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 59.73 (χ²/ndof = 1.2)      │              Nfcn = 46               │
│ EDM = 2.63e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.099   │   0.006   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.309e-6  │ 0.019e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬────────────────────────────────────────┐
│     │            a            A          tau │
├─────┼────────────────────────────────────────┤
│   a │     4.09e-05            0 17.98802e-12 │
│   A │            0            0            0 │
│ tau │ 17.98802e-12            0     3.46e-16 │
└─────┴────────────────────────────────────────┘

0.18813810101875328


In [65]:
print(len(data_2),'\n', 'rate di decadimenti muonici tra il 29genn-2febbr:', 1/(len(data_2)/((24*4)*3600)), 'Hz')
print (len(data_3),'\n', 'rate di decadimenti muonici tra il 2-5 febbraio:', 1/(len(data_3)/((24*3+1)*3600)), 'Hz')
print (len(data_4),'\n', 'rate di decadimenti muonici tra il 5-11 febbraio:', 1/(len(data_4)/((24*4+20)*3600)), 'Hz')

6139 
 rate di decadimenti muonici tra il 29genn-2febbr: 56.29581365043167 Hz
4577 
 rate di decadimenti muonici tra il 2-5 febbraio: 57.4175223945816 Hz
8759 
 rate di decadimenti muonici tra il 5-11 febbraio: 47.67667541956844 Hz


## Plot result + Discrepancy analysis

In [66]:
count,edges = np.histogram(data_refact , n_bins)


fig = go.Figure()
fig.add_trace(go.Bar(x=edges[:-1], y=count/len(data_refact), name='total', width=np.diff(edges)))

x_fit = np.linspace(min(data_refact) , max(data_refact) , n_bins)
y_fit = n.values["a"]*expon.pdf( x_fit , loc = n.values["A"] , scale = n.values["tau"])*np.diff(edges)[0]
fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name='fit', line=dict(shape='linear')))

In [67]:
# ho provato a vedere come fossero gli errori poissoniani usando i risultati dell'interpolazione come base

res = count/len(data_refact) - y_fit

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_fit, y=res, mode='markers', error_y=dict(array=np.sqrt(y_fit)), name='residuals'))
print(np.sqrt(y_fit))
fig.show()

[0.24498094 0.2379542  0.231129   0.22449957 0.21806029 0.21180571
 0.20573053 0.19982959 0.19409792 0.18853064 0.18312306 0.17787057
 0.17276874 0.16781325 0.16299989 0.1583246  0.1537834  0.14937246
 0.14508804 0.1409265  0.13688433 0.13295811 0.12914449 0.12544027
 0.12184229 0.11834751 0.11495297 0.11165579 0.10845319 0.10534245
 0.10232093 0.09938608 0.0965354  0.0937665  0.09107701 0.08846466
 0.08592725 0.08346261 0.08106867 0.07874339 0.07648481 0.07429101
 0.07216014 0.07009038 0.06807999 0.06612726 0.06423055 0.06238824
 0.06059877 0.05886062 0.05717233 0.05553247 0.05393964]


In [68]:
range_bins = np.arange(10 , 200 , 1)

chi2_list = []

for bin in range_bins:
    n = dataset_analysis( data_refact , exp , bins = bin , args=exp_args)
    n.migrad()
    n.hesse()
    chi2_list.append(1 - chi2.cdf(n.fval, df=n.ndof))

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=range_bins, y=chi2_list, mode="markers", name="chi squared"))
fig.add_trace(go.Scatter(x=[sturges(len(data_refact)), sturges(len(data_refact))],y=[min(chi2_list), max(chi2_list)],mode="lines",line=dict(dash="dash", color="red"),name="sturges value for binning"))
fig.update_layout(showlegend=True)
fig.show()